# SAL_W_WARD Deduplication

**Tess Vu**

Resolves duplicate EA_CODE rows in Jill's ArcGIS Pro spatial join output `sal_w_ward_new`.

Working CRS: EPSG:32735 (UTM 35S, meters), inherited from the input shapefile and preserved on output.

- Input:  `notebooks/data/sal_w_ward_new/sal_w_ward_new.shp`
- Output: `notebooks/data/sal_w_ward_dedup/sal_w_ward_dedup.shp` (38,380 rows, unique EA_CODE)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import pandas as pd
from src.io import ensure_dir
from src.paths import SAL_W_WARD_NEW, SAL_W_WARD_DEDUP, POP_PRED_FINAL

In [ ]:
sal_raw = gpd.read_file(SAL_W_WARD_NEW)

print(f"ORIGINAL: {sal_raw.shape[0]} features")
print(f"Unique EA_CODEs: {sal_raw['EA_CODE'].nunique()}")
print(f"Duplicate EA_CODE rows: {sal_raw.shape[0] - sal_raw['EA_CODE'].nunique()}")
print()
print(f"AREA column range: {sal_raw['AREA'].min():.2f} to {sal_raw['AREA'].max():.2f}")
print(f"PERCENTAGE column range: {sal_raw['PERCENTAGE'].min():.4f} to {sal_raw['PERCENTAGE'].max():.4f}")

ORIGINAL: 39177 features
Unique EA_CODEs: 38380
Duplicate EA_CODE rows: 797

AREA column range: 1550.18 to 623519119.33
PERCENTAGE column range: 26.4165 to 100.0001


In [ ]:
# CHECK 1: duplicate structure. Straddling SALs should appear exactly twice,
# 3+ occurrences point to a different cause.
dup_counts = sal_raw.groupby("EA_CODE").size()
n_appear_once  = (dup_counts == 1).sum()
n_appear_twice = (dup_counts == 2).sum()
n_appear_more  = (dup_counts > 2).sum()

print("DUPLICATE OCCURRENCE COUNTS")
print(f"  EA_CODEs appearing exactly once  : {n_appear_once}")
print(f"  EA_CODEs appearing exactly twice : {n_appear_twice}")
print(f"  EA_CODEs appearing 3 or more times: {n_appear_more}")
if n_appear_more > 0:
    print("  WARNING: EA_CODEs with 3+ rows (inspect these manually):")
    print(dup_counts[dup_counts > 2].head(10))

DUPLICATE OCCURRENCE COUNTS
  EA_CODEs appearing exactly once  : 37610
  EA_CODEs appearing exactly twice : 748
  EA_CODEs appearing 3 or more times: 22
EA_CODE
50410146.0    4
52410457.0    3
52910140.0    3
56810066.0    3
57510047.0    3
57510086.0    3
57510098.0    3
57510169.0    3
57510219.0    3
57510224.0    3
dtype: int64



In [ ]:
# CHECK 2: duplicates should carry two distinct WardIDs (SAL straddling two
# wards). Same-ward duplicates have a different cause.
dup_ea_codes = dup_counts[dup_counts > 1].index
dup_rows = sal_raw[sal_raw["EA_CODE"].isin(dup_ea_codes)].copy()

ward_col = "census_war"

if ward_col in dup_rows.columns:
    ward_per_dup = dup_rows.groupby("EA_CODE")[ward_col].nunique()
    same_ward = (ward_per_dup == 1).sum()
    diff_ward = (ward_per_dup > 1).sum()

    print("WARD ASSIGNMENT CHECK ACROSS DUPLICATE PAIRS")
    print(f"  Duplicate groups with different wards (expected): {diff_ward}")
    print(f"  Duplicate groups with same ward (unexpected):     {same_ward}")

    if same_ward > 0:
        same_ward_codes = ward_per_dup[ward_per_dup == 1].index[:5]
        print("  Sample same-ward duplicates (inspect these):")
        print(dup_rows[dup_rows["EA_CODE"].isin(same_ward_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])
else:
    print(f"WARNING: Ward column '{ward_col}' not found. Available columns:")
    print([c for c in sal_raw.columns if "ward" in c.lower() or "WAR" in c])

WARD ASSIGNMENT CHECK ACROSS DUPLICATE PAIRS
  Duplicate groups with different wards (expected): 0
  Duplicate groups with same ward (unexpected):     770
  Sample same-ward duplicates (inspect these):
        EA_CODE census_war          AREA  PERCENTAGE
68   50310126.0   52103004  8.946239e+06   99.999988
69   50310126.0   52103004  8.946239e+06   99.999988
85   50310193.0   52103003  2.679076e+06  100.000000
86   50310193.0   52103003  2.679076e+06  100.000000
172  50310099.0   52103019  1.020005e+06   93.826704
173  50310099.0   52103019  1.020005e+06   93.826704
197  50310063.0   52103012  1.026250e+07   90.546234
198  50310063.0   52103012  1.026250e+07   90.546234
256  50310086.0   52103018  1.189960e+06   99.387696
257  50310086.0   52103018  1.189960e+06   99.387696



In [ ]:
# DIAGNOSTIC: full inspection of EA_CODEs with 3+ occurrences.
high_dup_codes = dup_counts[dup_counts > 2].index
print(f"ALL EA_CODEs WITH 3+ OCCURRENCES: {len(high_dup_codes)} total")
print()

high_dup_rows = sal_raw[sal_raw["EA_CODE"].isin(high_dup_codes)].copy()
inspect_cols = ["EA_CODE", ward_col, "AREA", "PERCENTAGE"]

for ea_code, group in high_dup_rows.groupby("EA_CODE"):
    print(f"EA_CODE: {ea_code}  |  occurrences: {len(group)}  |  unique wards: {group[ward_col].nunique()}"
          f"  |  unique AREA values: {group['AREA'].nunique()}  |  unique PERCENTAGE values: {group['PERCENTAGE'].nunique()}"
          f"  |  unique geometry areas: {group.geometry.area.nunique()}")
    print(group[inspect_cols].to_string(index=True))
    print()

ALL EA_CODEs WITH 3+ OCCURRENCES: 22 total

EA_CODE: 50410146.0  |  occurrences: 4  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
        EA_CODE census_war          AREA  PERCENTAGE
286  50410146.0   52104002  1.240001e+08   99.935694
287  50410146.0   52104002  1.240001e+08   99.935694
288  50410146.0   52104002  1.240001e+08   99.935694
289  50410146.0   52104002  1.240001e+08   99.935694

EA_CODE: 52410457.0  |  occurrences: 3  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
         EA_CODE census_war          AREA  PERCENTAGE
1457  52410457.0   52502006  1.450021e+06       100.0
1458  52410457.0   52502006  1.450021e+06       100.0
1459  52410457.0   52502006  1.450021e+06       100.0

EA_CODE: 52910140.0  |  occurrences: 3  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
         EA_CODE census_war        

In [ ]:
# CHECK 3: AREA ties make the keep-first tie-break arbitrary.
tied = dup_rows[
    dup_rows.duplicated(subset=["EA_CODE", "AREA"], keep=False)
]["EA_CODE"].nunique()

print("AREA TIE CHECK")
print(f"  Duplicate EA_CODEs with tied AREA values: {tied}")
if tied > 0:
    tied_codes = dup_rows[
        dup_rows.duplicated(subset=["EA_CODE", "AREA"], keep=False)
    ]["EA_CODE"].unique()[:5]
    print("  Sample tied EA_CODEs (tie-break will be arbitrary):")
    print(dup_rows[dup_rows["EA_CODE"].isin(tied_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])

AREA TIE CHECK
  Duplicate EA_CODEs with tied AREA values: 770
  Sample tied EA_CODEs (tie-break will be arbitrary):
        EA_CODE census_war          AREA  PERCENTAGE
36   50310219.0   52103004  1.762411e+07   57.385730
37   50310219.0   52103004  1.762411e+07   57.385730
68   50310126.0   52103004  8.946239e+06   99.999988
69   50310126.0   52103004  8.946239e+06   99.999988
85   50310193.0   52103003  2.679076e+06  100.000000
86   50310193.0   52103003  2.679076e+06  100.000000
172  50310099.0   52103019  1.020005e+06   93.826704
173  50310099.0   52103019  1.020005e+06   93.826704
197  50310063.0   52103012  1.026250e+07   90.546234
198  50310063.0   52103012  1.026250e+07   90.546234



In [ ]:
# CHECK 4: geometry consistency — the SAL polygon should be identical across
# duplicate rows; area units are m² (EPSG:32735).
dup_rows["geom_area"] = dup_rows.geometry.area
geom_spread = dup_rows.groupby("EA_CODE")["geom_area"].agg(["min", "max"])
geom_spread["rel_diff"] = (geom_spread["max"] - geom_spread["min"]) / geom_spread["max"]

geom_mismatch = geom_spread[geom_spread["rel_diff"] > 0.01]

print("GEOMETRY CONSISTENCY CHECK (across duplicate pairs)")
print(f"  Duplicate groups with geometry area difference > 1%: {len(geom_mismatch)}")
if len(geom_mismatch) > 0:
    print("  WARNING: These EA_CODEs carry different geometries across duplicate rows:")
    print(geom_mismatch.head(10))
else:
    print("  All duplicate pairs carry consistent geometry.")

GEOMETRY CONSISTENCY CHECK (across duplicate pairs)
  Duplicate groups with geometry area difference > 1%: 0
  All duplicate pairs carry consistent geometry.



In [ ]:
# CHECK 5: retained WardID must match pop_pred_final, which used the same
# dominant-ward logic. Skipped if pop_pred_final.csv has not been built yet.
if POP_PRED_FINAL.exists():
    pop = pd.read_csv(POP_PRED_FINAL)
    pop_ward_lookup = pop.set_index("EA_CODE")["WardID"].astype(str)

    sal_sorted = sal_raw.sort_values("AREA", ascending=False)
    retained = sal_sorted.drop_duplicates(subset=["EA_CODE"], keep="first")[["EA_CODE", ward_col]].copy()
    retained = retained.rename(columns={ward_col: "shp_WardID"})
    retained["shp_WardID"] = retained["shp_WardID"].astype(str)
    retained["pop_WardID"] = retained["EA_CODE"].map(pop_ward_lookup)

    retained_dups = retained[retained["EA_CODE"].isin(dup_ea_codes)].copy()
    ward_matches    = (retained_dups["shp_WardID"] == retained_dups["pop_WardID"]).sum()
    ward_mismatches = (retained_dups["shp_WardID"] != retained_dups["pop_WardID"]).sum()

    print("WARD MATCH AGAINST pop_pred_final (duplicated EA_CODEs only)")
    print(f"  Retained WardID matches pop_pred_final: {ward_matches} / {len(retained_dups)}")
    print(f"  Mismatches: {ward_mismatches}")

    if ward_mismatches > 0:
        print()
        print("  Sample mismatches (shp_WardID vs pop_WardID):")
        print(retained_dups[retained_dups["shp_WardID"] != retained_dups["pop_WardID"]].head(10).to_string(index=False))
        print()
        print("  If mismatches are material, consider PERCENTAGE as the dedup key,")
        print("  or adopt WardID from pop_pred_final via a post-dedup merge.")
else:
    print(f"SKIPPED: {POP_PRED_FINAL} not found (run tess_newpred notebooks first).")

WARD MATCH AGAINST pop_pred_final (duplicated EA_CODEs only)
  Retained WardID matches pop_pred_final: 770 / 770
  Mismatches: 0



In [ ]:
# DEDUPLICATE: sort descending by AREA, keep first (dominant ward) per EA_CODE.
sal_dedup = (
    sal_raw
    .sort_values("AREA", ascending=False)
    .drop_duplicates(subset=["EA_CODE"], keep="first")
    .sort_index()
    .copy()
)

print(f"AFTER DEDUPLICATION: {sal_dedup.shape[0]} features")
print(f"Unique EA_CODEs: {sal_dedup['EA_CODE'].nunique()}")
print(f"Duplicates remaining: {sal_dedup.shape[0] - sal_dedup['EA_CODE'].nunique()}")

AFTER DEDUPLICATION: 38380 features
Unique EA_CODEs: 38380
Duplicates remaining: 0


In [ ]:
# POST-DEDUPLICATION: EA_CODE membership vs pop_pred_final, geometry validity, CRS.
if POP_PRED_FINAL.exists():
    pop_codes = set(pop["EA_CODE"].unique())
    sal_codes = set(sal_dedup["EA_CODE"].unique())

    print("POST-DEDUPLICATION EA_CODE MEMBERSHIP CHECK")
    print(f"  EA_CODEs in shapefile but NOT in pop_pred_final: {len(sal_codes - pop_codes)}")
    print(f"  EA_CODEs in pop_pred_final but NOT in shapefile: {len(pop_codes - sal_codes)}")

print()
print(f"  Null geometries : {sal_dedup.geometry.isna().sum()}")
print(f"  Empty geometries: {sal_dedup.geometry.is_empty.sum()}")
print(f"  CRS: {sal_dedup.crs}")

POST-DEDUPLICATION EA_CODE MEMBERSHIP CHECK
  EA_CODEs in shapefile but NOT in pop_pred_final: 0
  EA_CODEs in pop_pred_final but NOT in shapefile: 0

  Null geometries : 0
  Empty geometries: 0
  CRS: EPSG:32735


In [ ]:
output_path = SAL_W_WARD_DEDUP
ensure_dir(output_path.parent)
sal_dedup.to_file(output_path)

print(f"SAVED: {output_path}")
print(f"Final shape: {sal_dedup.shape}")

SAVED: data/sal_w_ward_dedup\sal_w_ward_dedup.shp
Final shape: (38380, 70)


## Notes

Duplicate rows carry identical ward, area, and geometry values, indicating a data artifact from the spatial join construction rather than a SAL-to-ward boundary overlap.